# Validation

## Setup

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"

%load_ext autoreload
%autoreload 2

## Upload LUT & NN

In [ ]:
from src.lut.lut import Lut
from src.nn.nn import NeuralNetwork

TRAIN_LUT_PATH = "models/luts/W_all/p3/lut_p3_W_all.nc"
VALIDATION_LUT_PATH = "models/luts/W_all/p3/lut_p3_validation.nc"
MODEL_NAME = "2L_32U_300E_relu_linear_cosdecay"
DIR_NN = f"models/nn/W_all/grid_search_p3/{MODEL_NAME}"

my_lut = Lut.load(TRAIN_LUT_PATH) # needed by save(lut=...) to keep _predictions.nc

validation_lut = Lut.load(VALIDATION_LUT_PATH)
validation_lut.print_info()

my_nn = NeuralNetwork.load(DIR_NN)
my_nn.print_info()

## Metrics & Plots

**Notation**: $y_i$ target value (LUT), $\hat y_i$ network prediction, $N$ number of samples,
$\bar y = \frac{1}{N}\sum_{i=1}^{N} y_i$ mean of the targets.

**Mean Squared Error** $:=\mathrm{MSE} = \frac{1}{N}\sum_{i=1}^{N} \left(y_i - \hat y_i\right)^2$

**Root Mean Squared Error** $:=\mathrm{RMSE} = \sqrt{\mathrm{MSE}} = \sqrt{\frac{1}{N}\sum_{i=1}^{N} \left(y_i - \hat y_i\right)^2}$

**Maximum Absolute Error** (worst case) $:=\max|\mathrm{err}| = \max_{i=1,\dots,N} \left|y_i - \hat y_i\right|$

**Coefficient of determination** $:=R^2 = 1 - \frac{\sum_{i=1}^{N}\left(y_i - \hat y_i\right)^2}{\sum_{i=1}^{N}\left(y_i - \bar y\right)^2}$

**Normalized fit** $:=\mathrm{FIT}\,[\%] = 100\left(1 - \frac{\lVert y - \hat y \rVert_2}{\lVert y - \bar y \rVert_2}\right)
= 100\left(1 - \sqrt{\frac{\sum_{i=1}^{N}\left(y_i - \hat y_i\right)^2}{\sum_{i=1}^{N}\left(y_i - \bar y\right)^2}}\right)$

In [ ]:
from IPython.display import display
from src.utils.metrics import metrics_table, TABLE_FORMATS

DIR_PLOTS = f"{DIR_NN}/plots"

if my_nn.lut_name != my_lut.lut_name:
    raise ValueError(f"{my_nn.nn_name}: trained on '{my_nn.lut_name}', not on '{my_lut.lut_name}'")

my_nn.validate(validation_lut)
my_nn.save(dir_nn=DIR_NN, lut=my_lut)

lut_metrics = my_lut.evaluate(validation_lut)
table = metrics_table({
    f"{MODEL_NAME}: training": my_nn.train_metrics,
    f"{MODEL_NAME}: validation": my_nn.validation_metrics,
    f"{my_lut.lut_name} interpolated: validation": lut_metrics,
})
display(table.style.format(TABLE_FORMATS))

my_lut.plot_parity(lut=validation_lut)
my_nn.plot_parity(lut=validation_lut,
                  save_path=f"{DIR_PLOTS}/{MODEL_NAME}_parity_validation.png")

In [ ]:
import numpy as np
MUSCLE = None          # BICshort; None = all the muscles

err = np.abs(my_lut._interp_grid(validation_lut) - validation_lut.activations)    # (n_W, A, V, T, M)
if MUSCLE is not None:
    err = err[..., MUSCLE:MUSCLE + 1]

axes = {"W": validation_lut.w, "angle": validation_lut.theta,
        "velocity": validation_lut.theta_dot, "torque_norm": validation_lut.tau_norm}
for ax, (name, values) in enumerate(axes.items()):
    others = tuple(i for i in range(err.ndim) if i != ax)            # the other axes and the muscles
    worst = np.nanmax(err, axis=others)
    rmse = np.sqrt(np.nanmean(err ** 2, axis=others))
    print(f"{name}:")
    for value, m, r in zip(values, worst, rmse):
        print(f"  {value:10.4g}: max {m:.4f} | RMSE {r:.4f}" + ("   <--" if m > 0.02 else ""))

In [ ]:
err_at = np.nanmax(err, axis=(0, 2, 4))                 # (angle, torque_norm): max over W, velocity, muscles
low = validation_lut.theta < 110
for label, mask in (("angles < 110", low), ("angles >= 110", ~low)):
    print(f"{label:>14}: max at the last 4 torque_norm =",
          np.round(err_at[mask][:, -4:].max(axis=0), 4))